In [98]:
import numpy as np 
import pandas as pd 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [99]:
train_raw = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test_raw = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

train = train_raw.copy()
test = test_raw.copy()
train.shape
train.info()
train.describe()
#check N/A
missing = train.isnull().sum().sort_values(ascending=False)
missing.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
Condition2         0
dtype: int64

In [100]:
#Handling missing values
nones = ['FireplaceQu',
    'GarageQual', 'GarageCond', 'GarageType', 'GarageFinish',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
for col in nones:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

medians = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea']
for col in medians:
    train[col] = train[col].fillna(train[col].median())
    test[col] = test[col].fillna(test[col].median())

modes = ['MasVnrType', 'Electrical']
for col in modes:
    train[col] = train[col].fillna(train[col].mode()[0])
    test[col] = test[col].fillna(test[col].mode()[0])

In [101]:
#Feature construction
#1. Total floor area
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']

#2. House age
train['HouseAge'] = train['YrSold'] - train['YearBuilt']
test['HouseAge'] = test['YrSold'] - test['YearBuilt']

#3. Reconstruction age
train['RemodAge'] = train['YrSold'] - train['YearRemodAdd']
test['RemodAge'] = test['YrSold'] - test['YearRemodAdd']

#4. Whether reconscturction
train['HasRemod'] = (train['YearRemodAdd'] != train['YearBuilt']).astype(int)
test['HasRemod'] = (test['YearRemodAdd'] != test['YearBuilt']).astype(int)

#5. Facilities
train['HasGarage'] = (train['GarageArea'] > 0).astype(int)
test['HasGarage'] = (test['GarageArea'] > 0).astype(int)

train['HasBsmt'] = (train['TotalBsmtSF'] > 0).astype(int)
test['HasBsmt'] = (test['TotalBsmtSF'] > 0).astype(int)

train['HasFireplace'] = (train['Fireplaces'] > 0).astype(int)
test['HasFireplace'] = (test['Fireplaces'] > 0).astype(int)

In [102]:
#Define model
y = np.log1p(train['SalePrice'])

train_features = train.drop('SalePrice', axis=1)
test_features = test.copy()

train_features = pd.get_dummies(train_features)
test_features = pd.get_dummies(test_features)
X, test_final = train_features.align(test_features, join='left', axis=1, fill_value=0)

In [103]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
#Ridge
ridge_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])

ridge_pipe.fit(X_train, y_train)
y_pred = ridge_pipe.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(rmse)

0.12520476373396006


In [104]:
alphas = [0.01, 0.1, 1, 10, 100]

for a in alphas:
    ridge = Ridge(a)
    ridge.fit(X_train, y_train)
    y_pred = ridge.predict(X_val)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    print(rmse)

0.12814706860242678
0.12780881102102803
0.13097019454447797
0.13599529654970488
0.14028163652185205


In [110]:
#Tried LASSO but no improvement
from sklearn.linear_model import Lasso
lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=0.1, max_iter=10000))
])

lasso_pipe.fit(X_train, y_train)

y_pred = lasso_pipe.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(rmse)
for a in alphas:
    lasso = Lasso(a)
    lasso.fit(X_train, y_train)
    y_pred = lasso.predict(X_val)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    print(rmse)
    coef = lasso.coef_
    print('Total:', len(coef))
    print('Selection:', np.sum(coef != 0))

0.23252460089306082
0.18628892843007616
Total: 305
Selection: 18
0.1901360331563799
Total: 305
Selection: 16
0.19114615303121835
Total: 305
Selection: 16
0.20181513791695602
Total: 305
Selection: 12
0.22142458907905538
Total: 305
Selection: 9
0.22595110708927382
Total: 305
Selection: 6
0.2322266522150783
Total: 305
Selection: 4
0.2513325471547795
Total: 305
Selection: 3
0.2694550677203278
Total: 305
Selection: 2
0.3046220872009764
Total: 305
Selection: 2


In [106]:
#Tried non-linear model but no improvement
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(rmse)

0.13197514419326545


In [107]:
#Using RidgeCV for optimalization
from sklearn.linear_model import RidgeCV
alphas = [0.75, 0.95, 1, 1.5, 3, 5, 10, 20, 50, 100] 
ridge_cv = RidgeCV(alphas = alphas, cv = 5)
ridge_cv.fit(X, y)
print(ridge_cv.alpha_)
best_alpha = ridge_cv.alpha_

ridge = Ridge(alpha=best_alpha)
ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(rmse)

10.0
0.13599529654970488


In [108]:
#Compare the result from RidgeCV to Ridge
# alpha = 0.1
ridge1 = Ridge(alpha=0.1)
ridge1.fit(X_train, y_train)
y_pred1 = ridge1.predict(X_val)
rmse1 = np.sqrt(mean_squared_error(y_val, y_pred1))

# alpha = 10
ridge2 = Ridge(alpha=10)
ridge2.fit(X_train, y_train)
y_pred2 = ridge2.predict(X_val)
rmse2 = np.sqrt(mean_squared_error(y_val, y_pred2))

print("alpha=0.1:", rmse1)
print("alpha=10 :", rmse2)

alpha=0.1: 0.12780881102102803
alpha=10 : 0.13599529654970488


In [111]:
# Final model
final_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=10))
])

final_model.fit(X, y)
test_pred_log = final_model.predict(test_final)
test_pred = np.expm1(test_pred_log)

submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': test_pred
})
submission.to_csv('submission.csv', index=False)
submission.head()

,Id,SalePrice
0,1461,121320.584076
1,1462,161544.434557
2,1463,181377.684056
3,1464,201199.223132
4,1465,193313.796960
